### Core libraries

In [ ]:
import random, gc, os

import numpy as np 

from collections import defaultdict

---
---
---
---
---
---
---
---
---
---

### Mass of a DM particle from Illustris_3

In [ ]:
m_DM = 0.0338857141965349   # * 10^10 M_Sun
n_DM = 455**3
l_grid = 75   # cMpc/h

---
---
---
---
---

### Numer of cores for multiprocessing

(Note: RAM is really the limiting factor.)

(Note: Below we offer the values used for an Apple M1 SIP with 16Gb of RAM.

In [ ]:
# The DTFE ones take quite a bit of RAM... that's why we use chunks to begin with, as they actually
#     roughly double the compute time. If you have 32Gb or more, perhaps try in one chunk.
ALL_cores_no___1_DTFE_KDTree           = 1
ALL_cores_no___1_DTFE_Density_Loop     = 1
ALL_cores_no___1_NGP                   = [4, 3, 3, 3]
ALL_cores_no___2_Smoothing             = [4, 3, 3, 3]
ALL_cores_no___3_Layer                 = [4, 4, 3, 3]
ALL_cores_no___4_Origins               = [4, 4, 3, 3]
ALL_cores_no___5_UOD_vals_and_lvls     = [4, 4, 2, 2]
ALL_cores_no___6_Levels_isolated_pairs = [4, 4, 4, 4]
ALL_cores_no___7_Finder                = [4, 4, 4, 4]

### DTFE

In [ ]:
grid_chunks  = 4      # the number of chunks along each axis to split the coordinates grid into
buffer_scale = 10.0   # a scaling factor for the buffer (i.e. how many tiems we take the original value over)

In [ ]:
parameter_median_filter = 3
parameter_grey_opening  = 3
parameter_grey_closing  = 3

### Wether to use for the fft (and going on) the relative density $\delta$ or the density $\rho$ for the grid.

In [ ]:
rho_delta = "delta"   # or, alternatively, "rho"

### Complete list of parameters

Take this example:

In [ ]:
# [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**5],["cdf"],[ True],[0.20,0.12],[2],[ True]],   # BASELINE



# [512]                                  - size - size of the grid

# [[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]]  - ecc - zoom-in region for each axis (default, i.e. max, size is [0,75] (cMpc/h))

# [ 0.0]                                 - Z - redshift

# [ 100]                                 - nnc - number of nearest centroids for the DTFE method (if == 0, then we use the NGP code)

# [0.15]                                 - R - smoothing radius used for Gaussian smoothing (sgm)

# [  10**5]                              - lvl - number of levels to discretize the grid into

# ["cdf"]                                - cl - "log"/"cdf" wether the grid is discretized using a density  logarithmic scale with equal number of cells in
#                                             each level or a density cdf one where the cdf is split into intervals and each one contains its elements

# [ True]                                - wether to look for origins in this type of file or not

# [0.20,0.12]                            - oud / [ud, od] - the density cdf interval before which all voids unite and after which all voids with origins higher than that
#                                             get merged into the rest

# [2]                                    - MK - which VFA to be used: 1-MK1 or 2-MK2
# [ True]                                - save1perc - save 1% files



# If the user desires to skip a step (past the DTFE/NGP ofc... otherwise why even add an empty list), just leave the [ ] entry empty.

---

In [ ]:
all_parameters = [
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[3*10**2],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[  10**3],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[3*10**3],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[  10**4],["cdf"],[ True],[0.100,0.02],[1],[False]],   # 1
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # BASELINE
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[3*10**4],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[  10**5],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[3*10**5],["cdf"],[ True],[          ],[ ],[     ]],   # lvl

    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[1.00],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[128],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[2.00],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R




    
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[3*10**2],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[  10**3],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[3*10**3],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[  10**4],["cdf"],[ True],[0.100,0.02],[1],[False]],   # 1
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # BASELINE
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[3*10**4],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[  10**5],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[3*10**5],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[  10**6],["cdf"],[ True],[          ],[ ],[     ]],   # lvl

    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.45],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[1.00],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[256],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[2.00],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R




    
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**3],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[3*10**3],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[1],[ True]],   # MK1
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[   0],[0.60],[       ],[     ],[     ],[          ],[ ],[     ]],   # no_centroids NGC
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[  10],[0.15],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # no_centroids
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[ True]],   # BASELINE
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[1000],[0.15],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # no_centroids
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["log"],[ True],[          ],[ ],[     ]],   # log
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[3*10**4],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**5],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[3*10**5],["cdf"],[ True],[          ],[ ],[     ]],   # lvl
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**6],["cdf"],[ True],[          ],[ ],[     ]],   # lvl

    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.30],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.45],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.60],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[1.00],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[2.00],[  10**4],["cdf"],[ True],[          ],[ ],[     ]],   # R

    
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.075,0.02],[2],[False]],   # Delta_CDF
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.01],[2],[False]],   # Delta_CDF
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.03],[2],[False]],   # Delta_CDF
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.10],[2],[False]],   # Delta_CDF
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.125,0.01],[2],[False]],   # Delta_CDF
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.125,0.02],[2],[False]],   # Delta_CDF
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.125,0.05],[2],[False]],   # Delta_CDF

    

    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.1],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.2],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.4],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  0.7],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  1.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  1.6],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  2.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  3.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  5.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  7.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[  9.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[ 12.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
    [[512],[[ 0.0,75.0],[ 0.0,75.0],[ 0.0,75.0]],[ 17.0],[ 100],[0.15],[  10**4],["cdf"],[ True],[0.100,0.02],[2],[False]],   # Z
                  ]

In [ ]:
### Summary of the above:

# no need for zooming-in.... 512 is already over the interparticle distance: all are [0.0,75.0]
# 
# 
# BASELINES:
# 128 | 0.0,75.0 | 0.0 | 100 | 0.60 | 10**4 | "cdf" | True | 0.20,0.12 | 2 | True
# 256 | 0.0,75.0 | 0.0 | 100 | 0.30 | 10**5 | "cdf" | True | 0.20,0.12 | 2 | True
# 512 | 0.0,75.0 | 0.0 | 100 | 0.15 | 10**5 | "cdf" | True | 0.20,0.12 | 2 | True
# 
# - MK1 -> 512 only | Z=0.0 only | same as BASELINE, but | run_origin=True | ... | ... | False
# 
# - each Z -> 512 only | same as BASELINE | save1perc=False
# - "log"  -> 512 only | same as BASELINE | save1perc=False
# - spread of lvls -> same as BASELINE, but | run_origin=True | ... | ... | False
#     - 128: 3*10**2, 10**3, 3*10**3, 10**4, 3*10**4, 10**5, 3*10**5
#     - 256: 3*10**2, 10**3, 3*10**3, 10**4, 3*10**4, 10**5, 3*10**5, 10**6
#     - 512:          10**3, 3*10**3, 10**4, 3*10**4, 10**5, 3*10**5, 10**6
# - no_centroids -> 512 only | Z=0.0 only | same as BASELINE, but | run_origin=True | ... | ... | False
#                -> 0, 10, 100, 1000
#                -> 0 uses R=0.60
# - different Rs -> Z=0 only | same as BASELINE, but | run_origin=True | ... | ... | False
#                - 128:                   0.60, 1.00, 2.00
#                - 256:       0.30, 0.45, 0.60, 1.00, 2.00
#                - 512: 0.15, 0.30, 0.45, 0.60, 1.00, 2.00
# 
# - defferent Delta_CDFs -> 512 only | Z=0 only | same as BASELINE, but | False
# 0.20,0.10 | 0.20,0.12 | 0.20,0.15 | 0.20,0.17 | 0.20,0.20 | 0.20,0.30 | 0.22,0.12 | 0.30,0.12

---

This has to be done by hand... but will save lots of time and clutter in the Analysis codes.

In [ ]:
BASELINES = [{"size":        128,
              "ecc":         [[0.0, 75.0], [0.0, 75.0], [0.0, 75.0]],
              "Z":          "00",
              "Z_float":     0.0,
              "Z_floatstr": "0.0",
              "Z_snapshot": "135",
              "nnc":         100,
              "R":           0.60,
              "lvl":         10**4,
              "cl":         "cdf",
              "uod":         [0.100, 0.02],
              "uod_str":    "[0.1_0.02]",
              "MK":         "MK2"},


             
             {"size":        256,
              "ecc":         [[0.0, 75.0], [0.0, 75.0], [0.0, 75.0]],
              "Z":          "00",
              "Z_float":     0.0,
              "Z_floatstr": "0.0",
              "Z_snapshot": "135",
              "nnc":         100,
              "R":           0.30,
              "lvl":         10**4,
              "cl":         "cdf",
              "uod":         [0.100, 0.02],
              "uod_str":    "[0.1_0.02]",
              "MK":         "MK2"},


             
             {"size":        512,
              "ecc":         [[0.0, 75.0], [0.0, 75.0], [0.0, 75.0]],
              "Z":          "00",
              "Z_float":     0.0,
              "Z_floatstr": "0.0",
              "Z_snapshot": "135",
              "nnc":         100,
              "R":           0.15,
              "lvl":         10**4,
              "cl":         "cdf",
              "uod":         [0.100, 0.02],
              "uod_str":    "[0.1_0.02]",
              "MK":         "MK2"}]

---
---
---

In [ ]:
ALL_sizes_and_cuts = []
for file in all_parameters:
    ALL_sizes_and_cuts.append([file[0][0], file[1]])

In [ ]:
#ALL_sizes_and_cuts = sorted([[t[0], list(t[1])] for t in set((sublist[0], tuple(tuple(inner) for inner in sublist[1])) for sublist in ALL_sizes_and_cuts)], key=lambda x: (x[0], x[1][0][0]))
ALL_sizes_and_cuts = sorted([[t[0], list(t[1])] for t in set((sublist[0], tuple(tuple(inner) for inner in sublist[1])) for sublist in ALL_sizes_and_cuts)], key=lambda x: (x[0], -x[1][0][1]))

In [ ]:
ALL_sizes_and_cuts = sorted([[t[0], list(t[1])] for t in set((sublist[0], tuple(tuple(inner) for inner in sublist[1])) for sublist in ALL_sizes_and_cuts)], key=lambda x: (-x[1][0][1], x[0]))

In [ ]:
ALL_sizes_and_cuts = [[_[0], [list(__) for __ in _[1]]] for _ in ALL_sizes_and_cuts]

In [ ]:
ALL_sizes = [_[0] for _ in ALL_sizes_and_cuts]

---

In [ ]:
# Please add the conversions to outher files if needed!

In [ ]:
conversion_zPaths = [  "00",   "01",   "02",  "04",  "07",  "10",  "16",  "20",  "30",  "50",  "70",  "90",  "120",  "170"]
conversion_znos   = [ "135",  "127",  "120", "108",  "95",  "85",  "73",  "68",  "60",  "49",  "41",  "35",   "28",   "21"]
conversion_ages   = [13.752, 12.462, 11.353, 9.463, 7.411, 5.977, 4.120, 3.356, 2.195, 1.205, 0.782, 0.560,  0.376,  0.231]

---

In [ ]:
ALL_Zs          = []; ALL_Zs_float    = []; ALL_Zs_floatstr = []; ALL_Zs_snapshot = []; ALL_ages = []
ALL_nncs        = []
ALL_R_cMpchs    = []
ALL_lvls        = []
ALL_cdf_logs    = []
ALL_run_Origins = []; ALL_uods        = []; ALL_run_levels  = []
ALL_run_MKs     = []; ALL_save1percs  = []

for i0 in range(len(ALL_sizes)):
    size     = ALL_sizes_and_cuts[i0][0]
    edge_cut = ALL_sizes_and_cuts[i0][1]
    ALL_Zs_float.append(   []); ALL_Zs_floatstr.append([]); ALL_Zs.append([]); ALL_Zs_snapshot.append([]); ALL_ages.append([])
    ALL_nncs.append(       [])
    ALL_R_cMpchs.append(   [])
    ALL_lvls.append(       [])
    ALL_cdf_logs.append(   [])
    ALL_run_Origins.append([]); ALL_uods.append([]); ALL_run_levels.append([])
    ALL_run_MKs.append(    []); ALL_save1percs.append([])
    
    for ap in all_parameters:
        if ap[0][0] == size and ap[1] == edge_cut:
            ALL_Zs_float[-1].append(ap[2][0])

    ALL_Zs_float[    -1] = np.unique(ALL_Zs_float[-1]).tolist()
    ALL_Zs_floatstr[ -1] = [str(_) for _ in ALL_Zs_float[-1]]
    ALL_Zs[          -1] = [_.replace(".", "") for _ in ALL_Zs_floatstr[-1]]
    ALL_Zs_snapshot[ -1] = [conversion_znos[conversion_zPaths.index(_)] for _ in ALL_Zs[-1]]
    ALL_ages[        -1] = [conversion_ages[conversion_zPaths.index(_)] for _ in ALL_Zs[-1]]

    for zPaths_no in ALL_Zs_float[-1]:
        ALL_nncs[       -1].append([])
        ALL_R_cMpchs[   -1].append([])
        ALL_lvls[       -1].append([])
        ALL_cdf_logs[   -1].append([])
        ALL_run_Origins[-1].append([]); ALL_uods[      -1].append([])
        ALL_run_levels[ -1].append([])
        ALL_run_MKs[    -1].append([]); ALL_save1percs[-1].append([])
        
        for ap in all_parameters:
            if ap[0][0] == size and ap[1] == edge_cut and ap[2][0] == zPaths_no:
                ALL_nncs[-1][-1].append(ap[3][0])
        ALL_nncs[-1][-1] = np.unique(ALL_nncs[-1][-1]).tolist()

        for NCs in ALL_nncs[-1][-1]:
            ALL_R_cMpchs[   -1][-1].append([])
            ALL_lvls[       -1][-1].append([])
            ALL_cdf_logs[   -1][-1].append([])
            ALL_run_Origins[-1][-1].append([]); ALL_uods[      -1][-1].append([])
            ALL_run_levels[ -1][-1].append([])
            ALL_run_MKs[    -1][-1].append([]); ALL_save1percs[-1][-1].append([])
            
            for ap in all_parameters:
                if ap[0][0] == size and ap[1] == edge_cut and ap[2][0] == zPaths_no and ap[3][0] == NCs:
                    if ap[4] != []: ALL_R_cMpchs[-1][-1][-1].append(ap[4][0])
            ALL_R_cMpchs[-1][-1][-1] = np.unique(ALL_R_cMpchs[-1][-1][-1]).tolist()
    
            for sigma_ps in ALL_R_cMpchs[-1][-1][-1]:
                ALL_lvls[       -1][-1][-1].append([])
                ALL_cdf_logs[   -1][-1][-1].append([])
                ALL_run_Origins[-1][-1][-1].append([]); ALL_uods[      -1][-1][-1].append([]); ALL_run_levels[-1][-1][-1].append([])
                ALL_run_MKs[    -1][-1][-1].append([]); ALL_save1percs[-1][-1][-1].append([])
                for ap in all_parameters:
                    if ap[0][0] == size and ap[1] == edge_cut and ap[2][0] == zPaths_no and ap[3][0] == NCs and ap[4][0] == sigma_ps:
                        if ap[5] != []: ALL_lvls[-1][-1][-1][-1].append(ap[5][0])
                ALL_lvls[-1][-1][-1][-1] = np.unique(ALL_lvls[-1][-1][-1][-1]).tolist()
    
                for lvl in ALL_lvls[-1][-1][-1][-1]:
                    ALL_cdf_logs[   -1][-1][-1][-1].append([])
                    ALL_run_Origins[-1][-1][-1][-1].append([]); ALL_uods[      -1][-1][-1][-1].append([]); ALL_run_levels[-1][-1][-1][-1].append([])
                    ALL_run_MKs[    -1][-1][-1][-1].append([]); ALL_save1percs[-1][-1][-1][-1].append([])
                    for ap in all_parameters:
                        if ap[0][0] == size and ap[1] == edge_cut and ap[2][0] == zPaths_no and ap[3][0] == NCs and ap[4][0] == sigma_ps and ap[5][0] == lvl:
                            if ap[6] != []: ALL_cdf_logs[-1][-1][-1][-1][-1].append(ap[6][0])
                    ALL_cdf_logs[-1][-1][-1][-1][-1] = np.unique(ALL_cdf_logs[-1][-1][-1][-1][-1]).tolist()
    
                    for cdf_log in ALL_cdf_logs[-1][-1][-1][-1][-1]:
                        ALL_run_Origins[-1][-1][-1][-1][-1].append(False); ALL_uods[      -1][-1][-1][-1][-1].append([]); ALL_run_levels[-1][-1][-1][-1][-1].append(False)
                        ALL_run_MKs[    -1][-1][-1][-1][-1].append([]);    ALL_save1percs[-1][-1][-1][-1][-1].append([])
                        for ap in all_parameters:
                            if ap[0][0] == size and ap[1] == edge_cut and ap[2][0] == zPaths_no and ap[3][0] == NCs and ap[4][0] == sigma_ps and ap[5][0] == lvl and ap[6][0] == cdf_log:
                                if ap[7] != []: ALL_run_Origins[-1][-1][-1][-1][-1][-1] = True
                                if ap[8] != []: ALL_uods[-1][-1][-1][-1][-1][-1].append(ap[8])
                        ALL_uods[-1][-1][-1][-1][-1][-1] = np.unique(ALL_uods[-1][-1][-1][-1][-1][-1], axis=0).tolist()
    
                        for Density_uod in ALL_uods[-1][-1][-1][-1][-1][-1]:
                            ALL_run_MKs[-1][-1][-1][-1][-1][-1].append([]); ALL_save1percs[-1][-1][-1][-1][-1][-1].append([])
                            for ap in all_parameters:
                                if ap[9] != []:
                                    if ap[0][0] == size and ap[1] == edge_cut and ap[2][0] == zPaths_no and ap[3][0] == NCs and ap[4][0] == sigma_ps and ap[5][0] == lvl and ap[6][0] == cdf_log and ap[8] == Density_uod:
                                        ALL_run_levels[-1][-1][-1][-1][-1][-1] = True
                                        ALL_run_MKs[   -1][-1][-1][-1][-1][-1][-1].append(ap[ 9][0])
                                        ALL_save1percs[-1][-1][-1][-1][-1][-1][-1].append(ap[10][0])
    
                            unique_args = np.unique(ALL_run_MKs[-1][-1][-1][-1][-1][-1][-1], return_index=True)[1]
                            ALL_run_MKs[   -1][-1][-1][-1][-1][-1][-1] = np.array(ALL_run_MKs[   -1][-1][-1][-1][-1][-1][-1])[unique_args].tolist()
                            ALL_save1percs[-1][-1][-1][-1][-1][-1][-1] = np.array(ALL_save1percs[-1][-1][-1][-1][-1][-1][-1])[unique_args].tolist()

---
---
---

### Edges of the cubes we're interested in [cMpc/h]

(Note: 0-75 is the default for the whole cube.)

(Note: Since this project uses data cubes in the following steps, we only allow for cubic spapes.)

In [ ]:
ALL_eccs = [[list(__) for __ in _[1]] for _ in ALL_sizes_and_cuts]

### Grid size (dxyz)

(Note: These are in units of ckpc/h, since this is how the files were given.)

In [ ]:
ALL_d_xyzs = [(ecc[0][1]-ecc[0][0])/size for ecc, size in zip(ALL_eccs, ALL_sizes)]

### Paths

In [ ]:
# Define the directory where your notebook is located
notebook_directory = "./"

# Create a directory to store the executed notebooks
output_directory = "./output/"

# Path to save files to
base_path = ".."

# Paths to snapshots abd location to save DTFE intermediary files to
base_path_snapshots = "../../../../../Volumes/Ulpia"
base_path_DTFE      = "../../../../../Volumes/Ulpia"


plots_path = base_path+"/Plots/"

ALL_file_paths = []
for z, size in enumerate(ALL_sizes):
    txt_ZONE = "__".join("_".join(str(ALL_eccs[z][i][j]) for j in range(len(ALL_eccs[z][i]))) for i in range(len(ALL_eccs[z])))
    txt = base_path+"/Modified_Data_"+str(int(size))+('' if (ALL_eccs[z] == [[0,75], [0,75], [0,75]]) else '___'+txt_ZONE) +"/"
    ALL_file_paths.append(txt)

---

### STD of the Gaussian smoothing in physical units [cMpc/h]

Those for all the current epoch and those for the it and all the previous ones (we use a fixed one for the previous frames, but if one desires different values, the code is easily adaptable).

In [ ]:
ALL_R_cells = []
for z, ALL_R_cMpchs_i in enumerate(ALL_R_cMpchs):
    d_xyz_sgm = ALL_d_xyzs[z]
    ALL_R_cells.append([])
    for ALL_R_cMpchs_ij in ALL_R_cMpchs_i:
        ALL_R_cells[-1].append([])
        for ALL_R_cMpchs_ijk in ALL_R_cMpchs_ij:
            ALL_R_cells[-1][-1].append([])
            for ALL_R_cMpchs_ijkl in ALL_R_cMpchs_ijk:
                ALL_R_cells[-1][-1][-1].append(round(ALL_R_cMpchs_ijkl/d_xyz_sgm, 3))  

---
---
---

### Number of permuations allowed in Finder

In [ ]:
times_we_tried_max = 2*3*4*5

### Random seed for replicating the results

As we later explain, even our code does contain parts where the seed definitely can play a crucial role.

In [ ]:
random.seed(9)

---
---
---